In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib

df = pd.read_csv('../data/raw/store-data-6aa6d7a3f171f140353680(1).csv')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10064 entries, 0 to 10063
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10064 non-null  int64  
 1   Order ID       10064 non-null  object 
 2   Order Date     10064 non-null  object 
 3   Ship Date      9963 non-null   object 
 4   Ship Mode      9763 non-null   object 
 5   Customer ID    10064 non-null  object 
 6   Customer Name  9713 non-null   object 
 7   Segment        10064 non-null  object 
 8   Country        10064 non-null  object 
 9   City           10064 non-null  object 
 10  State          10064 non-null  object 
 11  Postal Code    9864 non-null   object 
 12  Region         10064 non-null  object 
 13  Product ID     10064 non-null  object 
 14  Category       10064 non-null  object 
 15  Sub-Category   10064 non-null  object 
 16  Product Name   10064 non-null  object 
 17  Sales          9863 non-null   float64
 18  Quanti

In [2]:
df = df.drop_duplicates()

In [3]:
df[df.duplicated()]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [4]:
df.loc[df['Order Date'] > df['Ship Date'] ].head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.960,2.0,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.940,3.0,0.0,219.5820
5,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.860,7.0,0.0,14.1694
6,7,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.280,4.0,0.0,1.9656
7,8,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.152,6.0,0.2,90.7152


In [5]:
df.loc[df['Postal Code'].isnull()].head(5)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
53,54,CA-2016-105816,12/11/2016,12/17/2016,Standard Class,JM-15265,Janet Molinari,Corporate,United States,New York City,...,NaN,East,OFF-FA-10000304,Office Supplies,Fasteners,Advantus Push Pins,15.260,7.0,0.0,6.2566
75,76,US-2017-118038,12/9/2017,12/11/2017,First Class,KB-16600,Ken Brennan,Corporate,United States,Houston,...,NaN,Central,OFF-BI-10004182,Office Supplies,Binders,Economy Binders,1.248,3.0,0.8,-1.9344
93,94,CA-2015-149587,1/31/2015,2/5/2015,Second Class,KB-16315,Karl Braun,Consumer,United States,Minneapolis,...,NaN,Central,FUR-FU-10003799,Furniture,Furnishings,"Seth Thomas 13 1/2"" Wall Clock",53.340,3.0,0.0,16.5354
118,119,US-2015-136476,4/5/2015,4/10/2015,Standard Class,GG-14650,Greg Guthrie,Corporate,United States,Bristol,...,NaN,South,OFF-BI-10003650,Office Supplies,Binders,GBC DocuBind 300 Electric Binding Machine,157.794,1.0,0.7,-115.7156
141,142,CA-2017-106180,9/18/2017,9/23/2017,Standard Class,SH-19975,Sally Hughsby,Corporate,United States,San Francisco,...,NaN,West,OFF-AR-10000940,Office Supplies,Art,Newell 343,8.820,3.0,0.0,2.3814


**standardizing City names**

In [6]:
df['City'] = df['City'].str.title()

**Filling the missing Postal codes**

In [7]:
mode_dict = (
    df.dropna(subset=['Postal Code']).value_counts(['City', 'Postal Code']).reset_index().drop_duplicates('City').set_index('City')['Postal Code'].to_dict())


get_null_idx = df[df['Postal Code'].isna()].index

for idx in get_null_idx:
    city = df.at[idx, 'City']
    if city in mode_dict:
        df.at[idx, 'Postal Code'] = mode_dict[city]



df.loc[df['Postal Code'].isna()]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [8]:
df[df['Ship Mode'].isna()]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
28,29,US-2015-150630,9/17/2015,9/21/2015,NaN,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,...,19140.0,East,OFF-BI-10000474,Office Supplies,Binders,Avery Recycled Flexi-View Covers for Binding S...,9.618,2.0,0.7,-7.0532
94,95,CA-2015-149587,1/31/2015,2/5/2015,NaN,KB-16315,Karl Braun,Consumer,United States,Minneapolis,...,55407.0,Central,OFF-BI-10002852,Office Supplies,Binders,Ibico Standard Transparent Covers,32.960,2.0,0.0,16.1504
191,192,CA-2015-102281,10/12/2015,10/14/2015,NaN,MP-17470,Mark Packer,Home Office,United States,New York City,...,10035.0,East,OFF-PA-10000061,Office Supplies,Paper,Xerox 205,51.840,8.0,0.0,24.8832
252,253,CA-2016-146941,12/10/2016,12/13/2016,NaN,DL-13315,Delfina Latchford,Consumer,United States,New York City,...,10024.0,East,OFF-ST-10001228,Office Supplies,Storage,"Fellowes Personal Hanging Folder Files, Navy",80.580,6.0,0.0,22.5624
314,315,CA-2014-167850,8/9/2014,8/16/2014,NaN,AG-10525,Andy Gerbode,Corporate,United States,Saint Petersburg,...,33710.0,South,TEC-PH-10002398,Technology,Phones,AT&T 1070 Corded Phone,178.384,2.0,0.2,22.2980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9911,9912,US-2014-157231,4/5/2014,4/9/2014,NaN,RP-19855,Roy Phan,Corporate,United States,Richmond,...,40475.0,South,OFF-BI-10002852,Office Supplies,Binders,Ibico Standard Transparent Covers,115.360,7.0,0.0,56.5264
9964,9965,CA-2016-146374,12/5/2016,12/10/2016,NaN,HE-14800,Harold Engle,Corporate,United States,Newark,...,19711.0,East,FUR-FU-10002671,Furniture,Furnishings,Electrix 20W Halogen Replacement Bulb for Zoom...,13.400,1.0,0.0,6.4320
9993,9994,CA-2017-119914,5/4/2017,5/9/2017,NaN,CC-12220,Chris Cortes,Consumer,United States,Westminster,...,92683.0,West,OFF-AP-10002684,Office Supplies,Appliances,"Acco 7-Outlet Masterpiece Power Center, Wihtou...",243.160,2.0,0.0,72.9480
10016,4904,CA-2017-110884,3/7/2017,3/12/2017,NaN,SH-20395,NaN,Consumer,United States,New York City,...,10035.0,East,OFF-LA-10003510,office supplies,Labels,Avery 4027 File Folder Labels for Dot Matrix P...,91.590,3.0,0.0,42.1314


In [9]:
df[df['Customer ID'] == 'TB-21520']['Ship Mode'].mode()

0    Standard Class
Name: Ship Mode, dtype: object

**Filling the missing Ship mode**

In [10]:

mode_dict = (
    df.dropna(subset=['Ship Mode']).value_counts(['Customer ID', 'Ship Mode']).reset_index().drop_duplicates('Customer ID').set_index('Customer ID')['Ship Mode'].to_dict()
    )



get_null_idx = df[df['Ship Mode'].isna()].index

for idx in get_null_idx:
    customer = df.at[idx, 'Customer ID']
    if customer in mode_dict:
        df.at[idx, 'Ship Mode'] = mode_dict[customer]


In [11]:
df[df['Ship Mode'].isna()]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


**Filling the missing Customer Name**

In [12]:
df[df['Customer Name'].isna()]
# df[df['Customer ID'] == 'TB-21520']

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
29,30,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,NaN,Consumer,United States,Philadelphia,...,19140.0,East,FUR-FU-10004848,Furniture,Furnishings,"Howard Miller 13-3/4"" Diameter Brushed Chrome ...",124.200,3.0,0.2,15.5250
31,32,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,NaN,Consumer,United States,Philadelphia,...,19140.0,East,OFF-AR-10004042,Office Supplies,Art,"BOSTON Model 1800 Electric Pencil Sharpeners, ...",86.304,6.0,0.2,9.7092
33,34,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,NaN,Consumer,United States,Philadelphia,...,19140.0,East,OFF-AR-10001683,Office Supplies,Art,Lumber Crayons,15.760,2.0,0.2,3.5460
35,36,CA-2016-117590,12/8/2016,12/10/2016,First Class,GH-14485,NaN,Corporate,United States,Richardson,...,75080.0,Central,TEC-PH-10004977,Technology,Phones,GE 30524EE4,1097.544,7.0,0.2,123.4737
39,40,CA-2015-117415,12/27/2015,12/31/2015,Standard Class,SN-20710,NaN,Home Office,United States,Houston,...,77041.0,Central,FUR-CH-10004218,Furniture,Chairs,"Global Fabric Manager's Chair, Dark Gray",212.058,3.0,0.3,-15.1470
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10031,6743,US-2017-101784,7/6/2017,7/11/2017,Standard Class,PO-18850,NaN,Consumer,United States,Los Angeles,...,90008.0,West,FUR-CH-10001146,Furniture,Chairs,"Global Task Chair, Black",122.136,3.0,0.2,-13.7403
10033,9240,CA-2017-110310,10/27/2017,1/1/2010,First Class,NB-18655,NaN,Corporate,United States,Tallahassee,...,32303.0,South,OFF-PA-10001685,Office Supplies,Paper,Easy-staple paper,56.784,7.0,0.2,20.5842
10034,6120,CA-2017-143378,9/19/2017,9/25/2017,Standard Class,JR-16210,NaN,Corporate,United States,Springfield,...,97477.0,West,OFF-AR-10001915,Office Supplies,Art,Peel-Off China Markers,23.832,3.0,0.2,6.5538
10036,2996,US-2014-150532,7/14/2014,7/21/2014,Standard Class,PB-19150,NaN,Consumerr,United States,Phoenix,...,85023.0,West,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,55.920,5.0,0.2,6.2910


In [13]:
mode_dict = (
    df.dropna(subset=['Customer Name']).value_counts(['Customer ID', 'Customer Name']).reset_index().drop_duplicates('Customer ID').set_index('Customer ID')['Customer Name'].to_dict()
    )

print(mode_dict)


get_null_idx = df[df['Customer Name'].isna()].index

for idx in get_null_idx:
    customer = df.at[idx, 'Customer ID']
    if customer in mode_dict:
        df.at[idx, 'Customer Name'] = mode_dict[customer]

{'WB-21850': 'William Brown', 'JL-15835': 'John Lee', 'PP-18955': 'Paul Prost', 'JD-15895': 'Jonathan Doherty', 'CK-12205': 'Chloris Kastensmidt', 'MA-17560': 'Matt Abelman', 'EH-13765': 'Edward Hooks', 'SV-20365': 'Seth Vernon', 'EP-13915': 'Emily Phan', 'AP-10915': 'Arthur Prichep', 'Dp-13240': 'Dean percer', 'LC-16870': 'Lena Cacioppo', 'KL-16645': 'Ken Lonsdale', 'BM-11650': 'Brian Moss', 'GT-14710': 'Greg Tran', 'XP-21865': 'Xylona Preis', 'CL-12565': 'Clay Ludtke', 'KD-16495': 'Keith Dawkins', 'ZC-21910': 'Zuschuss Carroll', 'BF-11170': 'Ben Ferrer', 'SH-19975': 'Sally Hughsby', 'KM-16720': 'Kunst Miller', 'CB-12025': 'Cassandra Brandow', 'PG-18820': 'Patrick Gardner', 'DK-12835': 'Damala Kotsonis', 'CS-12250': 'Chris Selesnick', 'SC-20725': 'Steven Cartwright', 'NS-18640': 'Noel Staavos', 'CK-12595': 'Clytie Kelty', 'PK-19075': 'Pete Kriz', 'EA-14035': 'Erin Ashbrook', 'BH-11710': 'Brosina Hoffman', 'SJ-20125': 'Sanjit Jacobs', 'LA-16780': 'Laura Armstrong', 'LB-16795': 'Laurel 

In [14]:
# check for any other null Customer Name

df[df['Customer Name'].isna()]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
6505,6506,CA-2016-152331,6/26/2016,6/30/2016,Standard Class,LD-16855,NaN,Corporate,United States,Chicago,...,60653.0,Central,OFF-AR-10001547,Office Supplies,Art,Newell 311,5.304,3.0,0.2,0.4641


In [15]:
# df = df.dropna(subset=['Customer Name'])

In [16]:
z = df[df['Ship Date'].isna()]

# z.value_counts(['Customer ID' , 'City']).head(100)
# df[df['Customer ID'] == 'JM-15265']



In [17]:
(df['Order Date'] > df['Ship Date']).sum()

np.int64(1653)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10030 entries, 0 to 10063
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10030 non-null  int64  
 1   Order ID       10030 non-null  object 
 2   Order Date     10030 non-null  object 
 3   Ship Date      9929 non-null   object 
 4   Ship Mode      10030 non-null  object 
 5   Customer ID    10030 non-null  object 
 6   Customer Name  10029 non-null  object 
 7   Segment        10030 non-null  object 
 8   Country        10030 non-null  object 
 9   City           10030 non-null  object 
 10  State          10030 non-null  object 
 11  Postal Code    10030 non-null  object 
 12  Region         10030 non-null  object 
 13  Product ID     10030 non-null  object 
 14  Category       10030 non-null  object 
 15  Sub-Category   10030 non-null  object 
 16  Product Name   10030 non-null  object 
 17  Sales          9830 non-null   float64
 18  Quantity   

**standardise mixed date formats**

In [19]:
# Convert date from string into datetime

df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed')

In [20]:
# check the date type 
df[['Order Date', 'Ship Date']].dtypes

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object

In [21]:
df[df['Order Date'] > df['Ship Date']]

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1562,1563,US-2017-102890,2017-06-30,2010-01-01,Same Day,SG-20470,Sheri Gordon,Consumer,United States,New York City,...,10011.0,East,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1044.630,5.0,0.4,-295.9785
2517,2518,CA-2017-123134,2017-05-02,2010-01-01,Standard Class,DW-13585,Dorothy Wardle,Corporate,United States,Westfield,...,7090.0,East,FUR-FU-10003975,Furniture,Furnishings,Eldon Advantage Chair Mats for Low to Medium P...,129.930,3.0,0.0,12.9930
3279,3280,CA-2014-102988,2014-04-05,2010-01-01,Second Class,GM-14695,Greg Maxwell,Corporate,United States,Alexandria,...,22304.0,South,OFF-AR-10000127,Office Supplies,Art,Newell 321,22.960,7.0,0.0,6.6584
4939,4940,CA-2014-157147,2014-01-13,2010-01-01,Standard Class,BD-11605,Brian Dahlen,Consumer,United States,San Francisco,...,94109.0,West,OFF-AR-10003514,Office Supplies,Art,4009 Highlighters by Sanford,19.900,5.0,0.0,6.5670
7099,7100,CA-2015-124933,2015-12-26,2010-01-01,Second Class,DF-13135,David Flashing,Consumer,United States,New York City,...,10009.0,East,OFF-PA-10003302,Office Supplies,Paper,Xerox 1906,212.640,6.0,0.0,99.9408
9101,9102,CA-2015-163181,2015-11-07,2010-01-01,Standard Class,AB-10105,Adrian Barton,Consumer,United States,Houston,...,77041.0,Central,OFF-AR-10001683,Office Supplies,Art,Lumber Crayons,23.640,3.0,0.2,5.3190
10010,6753,CA-2016-154767,2016-06-28,2010-01-01,Second Class,BP-11155,Becky Pak,Consumer,United States,Paterson,...,7501.0,East,OFF-PA-10003039,Office Supplies,Paper,Xerox 1960,61.960,2.0,0.0,27.8820
10033,9240,CA-2017-110310,2017-10-27,2010-01-01,First Class,NB-18655,Nona Balk,Corporate,United States,Tallahassee,...,32303.0,South,OFF-PA-10001685,Office Supplies,Paper,Easy-staple paper,56.784,7.0,0.2,20.5842


**fix date Same Day**

In [22]:
df.loc[df["Ship Mode"] == 'Same Day', 'Ship Date'] = df['Order Date']

In [23]:
df['Ship Mode'].value_counts()

Ship Mode
Standard Class    6014
Second Class      1936
First Class       1544
Same Day           536
Name: count, dtype: int64

In [24]:
# check how many unique Shipping mode are in the dataframe

df['Ship Mode'].unique()


array(['Second Class', 'Standard Class', 'First Class', 'Same Day'],
      dtype=object)

In [25]:
df["Ship Duration"] = df['Ship Date'] - df["Order Date"]

In [26]:
df.head(10)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Ship Duration
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136,3 days
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820,3 days
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714,4 days
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,7 days
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164,7 days
5,6,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7.0,0.00,14.1694,5 days
6,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800,4.0,0.00,1.9656,5 days
7,8,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6.0,0.20,90.7152,5 days
8,9,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3.0,0.20,5.7825,5 days
9,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5.0,0.00,34.4700,5 days


**Fix the dates inconsistencies**

In [27]:
mode_dict = (
    df.value_counts(['Ship Mode', 'Ship Duration']).reset_index().drop_duplicates('Ship Mode').set_index('Ship Mode')['Ship Duration'].to_dict()
    )

wrong_date_idx = df.loc[(df['Order Date'] > df['Ship Date']) | (df['Ship Date'].isna())].index

for i in wrong_date_idx:
    if df.at[i, 'Ship Mode'] in mode_dict:
        df.at[i, 'Ship Date'] = (
            df.at[i, 'Order Date']
            + mode_dict[df.at[i, 'Ship Mode']]
        )


df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 10030 entries, 0 to 10063
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype          
---  ------         --------------  -----          
 0   Row ID         10030 non-null  int64          
 1   Order ID       10030 non-null  object         
 2   Order Date     10030 non-null  datetime64[ns] 
 3   Ship Date      10030 non-null  datetime64[ns] 
 4   Ship Mode      10030 non-null  object         
 5   Customer ID    10030 non-null  object         
 6   Customer Name  10029 non-null  object         
 7   Segment        10030 non-null  object         
 8   Country        10030 non-null  object         
 9   City           10030 non-null  object         
 10  State          10030 non-null  object         
 11  Postal Code    10030 non-null  object         
 12  Region         10030 non-null  object         
 13  Product ID     10030 non-null  object         
 14  Category       10030 non-null  object         
 15  Sub-Cat

**Make sure there is no typo in  Segment**

In [28]:

idx_segment = df.loc[df['Segment'] == 'Consumerr'].index

for i in idx_segment:
    df.at[i, 'Segment'] = 'Consumer'

df['Segment'].value_counts()


Segment
Consumer       5206
Corporate      3003
Home Office    1778
Corporrate       26
Home Ofice       17
Name: count, dtype: int64

**Check if there is any Product with wrong Product ID**

In [29]:
# use Title to standardize product name

df['Product Name'] = df['Product Name'].str.title()


In [30]:
# group by product name and sotre them in dic

mode_dict = (
    df.value_counts(['Product Name', 'Product ID']).reset_index().drop_duplicates('Product Name').set_index('Product Name')['Product ID'].to_dict()
    )
    
for i in df.index:
    if df.at[i, 'Product Name'] in mode_dict:
        df.at[i, 'Product ID'] = mode_dict[df.at[i, 'Product Name']]


**Hash customer name via library hashlib SHA-256**

In [31]:

#reset of null Custome Names to 'Unknown'

df.loc[df['Customer Name'].isna(), 'Customer Name'] = 'Unknown'

In [32]:
df['Customer Name'] = df['Customer Name'].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)

**Correct the Sales-------------------------------------**

In [33]:
df[['Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']]

,Product Name,Sales,Quantity,Discount,Profit
0,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,Self-Adhesive Address Labels For Typewriters B...,14.6200,2.0,0.00,6.8714
3,Bretford Cr4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164
...,...,...,...,...,...
10059,Geemarc Amplipower60,278.4000,3.0,0.00,80.7360
10060,Safco Industrial Shelving,73.8500,1.0,0.00,2.2155
10061,Easy-Staple Paper,35.4400,1.0,0.00,16.6568
10062,"Global Push Button Manager'S Chair, Indigo",48.7120,1.0,0.20,5.4801


**Get the price per unit**

In [34]:
df_clean = df[df['Sales'].notna() & df['Quantity'].notna() & (df['Quantity'] > 0) & (df['Discount'].between(0, 1))].copy()


#? UnitPrice = Sales / (Quantity × ( 1 − Discount))

df_clean['UnitPrice'] = ( df_clean['Sales'] / (df_clean['Quantity'] * (1 - df_clean['Discount'])))

mode_p = df_clean.value_counts(['Product ID', 'UnitPrice']).reset_index().drop_duplicates('Product ID').set_index('Product ID')['UnitPrice'].to_dict()

df['UnitPrice'] = df['Product ID'].map(mode_p)



**drop the unique null products that only apeared once "7 rows"**

In [35]:
df.dropna(subset='UnitPrice', inplace=True)

**Validate and correct Sales values**

In [36]:
clone_df = df[df['Sales'].notna() & df['Quantity'].notna() & (df['Quantity'] > 0) & (df['Discount'].between(0, 1))].copy()

def calculate_solds(df, clone_df):
    clone_df['Sales'] = clone_df['Sales'].round(2)
    wrong_value = clone_df.loc[clone_df['Sales'] != ( clone_df['UnitPrice'] * clone_df['Quantity'] * (1 - clone_df['Discount'])).round(2)].index


    #? sales = unitprice  * quantity * (1 - discount)

    for i in wrong_value:
        df.at[i, 'Sales'] = (df.at[i, 'UnitPrice'] * df.at[i, 'Quantity'] * (1 - df.at[i, 'Discount'])).round(2)

calculate_solds(df, clone_df)


**Validate and correct inconsistent discount values**

In [37]:
#? discount = 1 - (sales/ (unitprice * quantity))

clone_2 = df[df['Sales'].notna() & df['Quantity'].notna() & (df['Quantity'] > 0)].copy()


wrong_discount = clone_2.loc[clone_2['Discount'].round(2) != (1 - (clone_2['Sales'] / (clone_2['UnitPrice'] * clone_2['Quantity']))).round(2)].index

for i in wrong_discount:
    df.at[i, 'Discount'] = (1 - (df.at[i, 'Sales'] / (df.at[i, 'UnitPrice'] * df.at[i, 'Quantity']))).round(2)

**Refill the null Sales after discount fix**

In [38]:
clone_df = df[df['Quantity'].notna() & (df['Quantity'] > 0)].copy()


# call function calculate_solds and pass it cleaned df
calculate_solds(df, clone_df)


In [39]:
# check for any extra null Sales

df[df['Sales'].isna()]


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Ship Duration,UnitPrice
345,346,CA-2017-169901,2017-06-15,2017-06-19,Standard Class,CC-12550,2ff69240af214a8e97793e0a4107d4b100b73199f22c33...,Consumer,United States,San Francisco,...,TEC-PH-10002293,Technology,Phones,Anker 36W 4-Port Usb Wall Charger Travel Power...,NaN,NaN,0.2,4.7976,4 days,19.99
347,348,CA-2017-134306,2017-07-08,2017-07-12,Standard Class,TD-20995,7501160fd5e8d90c7f1d80d59fc98e9efda931f6889771...,Consumer,United States,Lowell,...,OFF-PA-10000474,Office Supplies,Paper,Easy-Staple Paper,NaN,NaN,0.0,11.5432,4 days,35.44
2877,2878,CA-2016-152072,2016-01-15,2016-01-19,Standard Class,Dp-13240,c0eac2e6ca018856504d35f3f5654e9e43022df62bab2f...,Home Office,United States,Westfield,...,OFF-EN-10003040,Office Supplies,Envelopes,Quality Park Security Envelopes,NaN,NaN,0.0,24.5998,4 days,26.17
5894,5895,CA-2017-100825,2017-09-09,2017-09-14,Standard Class,KD-16495,059ae84db3a4e7d61daf4943cf7c7353a89541e58541d2...,Corporate,United States,Los Angeles,...,OFF-ST-10003123,Office Supplies,Storage,Fellowes Bases And Tops For Staxonsteel/High-S...,NaN,NaN,0.0,23.9688,5 days,33.29


**Fill the missing Quantity values**

In [40]:
clean_quantity = df[(df['Quantity'].notna()) & (df['Sales'].notna())].copy()

# create a dic to store each product and its most friquent quantity

mode_quantity = clean_quantity.value_counts(['Product ID', 'Quantity']).reset_index().drop_duplicates('Product ID').set_index('Product ID')['Quantity'].to_dict()


empty_quantities = df.loc[(df['Quantity'].isna() ) | (df['Quantity'] < 0)].index

for i in empty_quantities:
    df.at[i, 'Quantity'] = mode_quantity[df.at[i, 'Product ID']]


In [41]:
# Calculate the rest of sales that are null

wrong_value = df.loc[df['Sales'] != ( df['UnitPrice'] * df['Quantity'] * (1 - df['Discount'])).round(2)].index

#? sales = unitprice  * quantity * (1 - discount)

for i in wrong_value:
    df.at[i, 'Sales'] = (df.at[i, 'UnitPrice'] * df.at[i, 'Quantity'] * (1 - df.at[i, 'Discount'])).round(2)

In [42]:
# Drop the extra columns
df.drop(columns=['Profit'], inplace=True)
df.columns


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Ship Duration',
       'UnitPrice'],
      dtype='object')

In [43]:

df['Quantity'] = df['Quantity'].astype(int)
df.to_csv('../data/processed/cleaned_data.csv', index=False)